# Week 3 – Text Preprocessing (1/2)
**Notebook 1 — Text Preprocessing Process**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son (hojinson@hanyang.ac.kr)

In this notebook we practice the **lexical analysis** steps introduced in the lecture *Text Preprocessing*:

| Step | Lecture concept | NLTK tool |
|---|---|---|
| 1 | Sentence splitting | `sent_tokenize` |
| 2 | Tokenization | `word_tokenize`, `WordPunctTokenizer`, `RegexpTokenizer` |
| 3 | Noise / stop-word removal | `nltk.corpus.stopwords` |
| 4 | Morphological analysis (stemming, lemmatization) | `PorterStemmer`, `LancasterStemmer`, `WordNetLemmatizer` |
| 5 | Part-of-Speech (POS) tagging | `nltk.pos_tag` |
| 6 | Named Entity Recognition (NER) | `nltk.ne_chunk` |

> 💡 **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom (`Shift + Enter`).

## &nbsp;0. Setup
[NLTK (Natural Language Toolkit)](https://www.nltk.org/) is already installed in Google Colab.
However, the **language resources** (tokenizer models, WordNet, stop-word lists, taggers, …) must be downloaded once per session.

> ⚠️ Since NLTK 3.9 the tokenizer / tagger resources have new names ending with `_tab` or `_eng`
> (`punkt_tab`, `averaged_perceptron_tagger_eng`, `maxent_ne_chunker_tab`, `tagsets_json`).
> If you see a `LookupError` later, come back to this cell and check that the missing resource is listed here.

In [ ]:
import nltk
print("NLTK version:", nltk.__version__)

resources = [
    'punkt', 'punkt_tab',                                        # sentence / word tokenizers
    'stopwords',                                                 # stop-word lists
    'wordnet', 'omw-1.4',                                        # lemmatizer dictionary
    'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',  # POS tagger
    'tagsets', 'tagsets_json',                                   # POS tag descriptions (nltk.help)
    'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words',       # named entity chunker
]
for r in resources:
    nltk.download(r, quiet=True)
print("All resources downloaded.")

## &nbsp;1. Sentence Splitting
The lecture pointed out that a sentence boundary is **not** simply "every period":
`"Mr. X"`, `"3.14"`, `"U.S."` contain periods that do *not* end a sentence.
NLTK's `sent_tokenize` uses the **Punkt** model, which was trained to recognise abbreviations and such cases.

In [ ]:
para = "Hello everyone. It's good to see you. Let's start our Unstructured Data Analysis class!"

from nltk.tokenize import sent_tokenize
print(sent_tokenize(para))
# Splits on sentence-final punctuation such as . ! ?  (period, exclamation mark, question mark)

In [ ]:
# A harder case: abbreviations and decimal numbers should NOT split the sentence.
hard = "Dr. Kim paid $3.50 for coffee in the U.S. yesterday. Then she went home."
print(sent_tokenize(hard))

In [ ]:
# Punkt models exist for several languages. Pass the language name instead of loading a .pickle file
# (the old `nltk.data.load('tokenizers/punkt/french.pickle')` style is deprecated).
paragraph_french = """Je t'ai demandé si tu m'aimais bien, Tu m'a répondu non.
Je t'ai demandé si j'étais jolie, Tu m'a répondu non.
Je t'ai demandé si j'étai dans ton coeur, Tu m'a répondu non."""

print(sent_tokenize(paragraph_french, language='french'))

In [ ]:
para_kor = "안녕하세요, 여러분. 만나서 반갑습니다. 이제 비정형데이터분석 수업을 시작해봅시다!"
print(sent_tokenize(para_kor))   # The punctuation-based sentence splitter also works reasonably for Korean.

## &nbsp;2. Word Tokenization
Text is split into basic units called **tokens** (word tokens, number tokens, punctuation tokens, …).
Even tokenization can be difficult: *Is `John's` one token or two? What about `can't`, `database` vs. `data base`, `C++`?*
Different tokenizers make different decisions, so let's compare three of them.

### &nbsp;2-1. word_tokenize vs WordPunctTokenizer

In [ ]:
from nltk.tokenize import word_tokenize, WordPunctTokenizer

print("word_tokenize      :", word_tokenize(para))
print("WordPunctTokenizer :", WordPunctTokenizer().tokenize(para))

* `word_tokenize` handles contractions "naturally": `It's` → `It`, `'s` and `don't` → `do`, `n't`.
* `WordPunctTokenizer` separates **every** punctuation mark: `It's` → `It`, `'`, `s`.

Choose the tokenizer according to the text you analyse: `word_tokenize` is a good default for natural English text, while
`WordPunctTokenizer` is useful when you want punctuation separated explicitly.

In [ ]:
print(word_tokenize(para_kor))   # For Korean, only spaces/punctuation are used → particles (조사) stay attached to nouns.

### &nbsp;2-2. Regular Expressions
`RegexpTokenizer` lets **you** decide what a token is by giving a regular-expression pattern.

| Pattern | Meaning |
|---|---|
| `\w` | a word character: letter, digit or `_` (`[a-zA-Z0-9_]`) |
| `[\w']` | a word character **or** an apostrophe |
| `+` | one or more repetitions |
| `{3,}` | three or more repetitions |

> Use a **raw string** (`r"..."`) for regular expressions so that `\w` is not interpreted as a Python escape sequence.

In [ ]:
from nltk.tokenize import RegexpTokenizer

text1 = "Sorry, I can't go there."

tokenizer = RegexpTokenizer(r"[\w']+")     # letters, digits and apostrophes  → "can't" stays one token
print(tokenizer.tokenize(text1))

In [ ]:
tokenizer = RegexpTokenizer(r"[\w]+")      # apostrophe is NOT allowed → "can't" is split into "can" and "t"
print(tokenizer.tokenize(text1))

In [ ]:
tokenizer = RegexpTokenizer(r"[\w']{3,}")  # only tokens with 3 or more characters → "I" is dropped
print(tokenizer.tokenize(text1.lower()))   # .lower() converts the text to lowercase before tokenization

**❓ Quick check** – Which pattern would you use for tweets such as `"@user I loooove #Borderlands3 !!! http://t.co/abc"`?
Try to write a pattern that keeps `#Borderlands3` as one token and drops the URL. (We will need this idea in the assignment.)

In [ ]:
tweet = "@user I loooove #Borderlands3 !!! http://t.co/abc"

import re
tweet_no_url = re.sub(r"http\S+", " ", tweet)          # remove URLs first
print(RegexpTokenizer(r"[#@]?\w+").tokenize(tweet_no_url))   # keep hashtags / mentions as a single token

## &nbsp;3. Stop-words
**Stop-words** (`the`, `is`, `and`, …) appear in almost every document, so they carry little information for most text-mining tasks.
NLTK provides ready-made stop-word lists for many languages. You can also define **custom** stop-words for your domain (this is the usual approach for Korean).

In [ ]:
from nltk.corpus import stopwords

english_stops = set(stopwords.words('english'))   # a set → fast membership test, no duplicates
print(len(english_stops), "English stop-words, e.g.:", sorted(english_stops)[:15])

In [ ]:
text1 = "Sorry, I couldn't go to the movie yesterday."

tokenizer = RegexpTokenizer(r"[\w']+")
tokens = tokenizer.tokenize(text1.lower())         # 1) lowercase  2) tokenize

result = [word for word in tokens if word not in english_stops]   # 3) keep only non-stop-words
print("before:", tokens)
print("after :", result)

In [ ]:
# Custom stop-words (useful for Korean or for domain-specific words)
my_stopwords = ['sorry', 'yesterday']
result = [word for word in result if word not in my_stopwords]
print(result)

**❓ Why is lowercasing done *before* stop-word removal?**
The stop-word list is lowercase (`'the'`, not `'The'`). Lowercasing first makes the match case-insensitive and consistent across tokens.

## &nbsp;4. Stemming & Lemmatization
Words are changed by **inflection** (`car → cars`, `give → gives, gave, given`). Two approaches reduce them to a common form:

| | Stemming | Lemmatization |
|---|---|---|
| Idea | cut off suffixes with rules | look up the dictionary form (*lemma*) |
| Output | a *stem* that may not be a real word (`comput`) | an actual word (`compute`) |
| Speed / cost | fast, simple | slower, needs a lexicon (WordNet) and often the POS |
| Typical use | Information Retrieval | Text mining where semantics matter |

### &nbsp;4-1. Stemming

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer

porter = PorterStemmer()
lancaster = LancasterStemmer()

words = ['cooking', 'cookery', 'cookbooks', 'computers', 'stocks', 'stockings', 'army', 'arm']
print(f"{'word':<12}{'Porter':<12}{'Lancaster':<12}")
for w in words:
    print(f"{w:<12}{porter.stem(w):<12}{lancaster.stem(w):<12}")

* **Porter** stemmer – the standard rule-based algorithm for English, relatively conservative.
* **Lancaster** stemmer – more aggressive; stems are shorter and less recognisable.

Notice the disadvantages from the lecture: `computers → comput` (not a word), `army → armi` / `arm → arm` (different words map to different stems while `stocks`/`stockings` may collide).

In [ ]:
para = "Hello everyone. It's good to see you. Let's start our text mining class!"
tokens = word_tokenize(para)
print(tokens)
print([porter.stem(t) for t in tokens])   # stemming every token

### &nbsp;4-2. Lemmatization
`WordNetLemmatizer` returns the dictionary form. It assumes every word is a **noun** unless you pass the part of speech (`pos='v'`, `'a'`, `'r'`).

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize('cooking'))            # treated as a noun → unchanged
print(lemmatizer.lemmatize('cooking', pos='v'))   # as a verb → 'cook'
print(lemmatizer.lemmatize('cookery'))
print(lemmatizer.lemmatize('cookbooks'))          # plural noun → singular

In [ ]:
# Stemming vs. lemmatization on the same word
for w in ['believes', 'better', 'studies', 'running']:
    print(f"{w:<10} stem: {porter.stem(w):<8} lemma(n): {lemmatizer.lemmatize(w):<8} lemma(v): {lemmatizer.lemmatize(w, pos='v'):<8} lemma(a): {lemmatizer.lemmatize(w, pos='a')}")

## &nbsp;5. POS Tagging
Given a token sequence, a POS tagger predicts the grammatical category of each token (noun, verb, adjective, …).
The same token can receive **different tags depending on context**: *They love this game* → `love` is a verb, *We need more love* → `love` is a noun.

NLTK's default English tagger uses the **Penn Treebank** tag set (`NN`, `VB`, `JJ`, …).

In [ ]:
tokens = word_tokenize("Hello everyone. It's good to see you. Let's start our text mining class!")
print(nltk.pos_tag(tokens))

In [ ]:
# The same word, different POS depending on context
print(nltk.pos_tag(word_tokenize("They love this game.")))            # love → VBP (verb)
print(nltk.pos_tag(word_tokenize("We need more love in the world.")))  # love → NN  (noun)

# Taggers are statistical (~96-97 % accuracy) – they make mistakes on unusual sentences.
print(nltk.pos_tag(word_tokenize("Time flies like an arrow.")))       # 'flies' should be a verb here!

In [ ]:
# What does a tag mean?  nltk.help.upenn_tagset('<TAG>') prints its definition and examples.
nltk.help.upenn_tagset('NN')
nltk.help.upenn_tagset('VB')
nltk.help.upenn_tagset('JJ')

In [ ]:
# Keep only the tokens whose tag is in a chosen set (e.g. nouns / verbs / adjectives)
my_tag_set = ['NN', 'VB', 'JJ']
my_words = [word for word, tag in nltk.pos_tag(tokens) if tag in my_tag_set]
print(my_words)

In [ ]:
# Attach the tag to the token, e.g. 'class/NN' – a common way to keep word and POS together
words_with_tag = ['/'.join(item) for item in nltk.pos_tag(tokens)]
print(words_with_tag)

### &nbsp;5-1. POS-aware Lemmatization
`WordNetLemmatizer` needs the WordNet POS (`n`, `v`, `a`, `r`), whereas `pos_tag` returns Penn Treebank tags (`NN`, `VBD`, `JJ`, `RB`, …).
The helper below maps the **first letter** of a Penn tag to the WordNet POS. This pattern will be reused in the assignment.

In [ ]:
from nltk.corpus import wordnet

def penn_to_wordnet(tag):
    if tag.startswith('J'):   return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('N'): return wordnet.NOUN
    elif tag.startswith('R'): return wordnet.ADV
    else:                     return None   # e.g. determiners, prepositions

sentence = "The striped bats were hanging on their feet and ate the best fishes"
tagged = nltk.pos_tag(word_tokenize(sentence))

lemmas = []
for word, tag in tagged:
    wn_tag = penn_to_wordnet(tag)
    lemmas.append(lemmatizer.lemmatize(word, wn_tag) if wn_tag else word)

print("tagged :", tagged)
print("lemmas :", lemmas)

## &nbsp;6. Named Entity Recognition
NER locates and classifies text spans into pre-defined categories such as **PERSON**, **ORGANIZATION**, **GPE** (geo-political entity: countries, cities), dates, monetary values, …

NLTK's `ne_chunk` is a classifier-based chunker. It needs **POS-tagged** input, so the pipeline is:
`word_tokenize` → `pos_tag` → `ne_chunk`.

In [ ]:
from nltk import word_tokenize, pos_tag, ne_chunk

sentence = "James is working at Disney in London"

tokenized_sentence = word_tokenize(sentence)   # 1) tokenize
tagged_sentence = pos_tag(tokenized_sentence)  # 2) POS tagging (required by ne_chunk)
print(tagged_sentence)

In [ ]:
ner_sentence = ne_chunk(tagged_sentence)       # 3) named entity chunking
print(ner_sentence)

In [ ]:
# Extract only the named entities from the tree as (entity, label) pairs
def extract_entities(tree):
    entities = []
    for subtree in tree:
        if hasattr(subtree, 'label'):              # a chunk such as (PERSON James/NNP)
            entity = " ".join(word for word, tag in subtree.leaves())
            entities.append((entity, subtree.label()))
    return entities

print(extract_entities(ner_sentence))
print(extract_entities(ne_chunk(pos_tag(word_tokenize("Elon Musk visited Seoul and met Samsung executives in October")))))

## &nbsp;7. Full Pipeline
Let's chain everything into one function. This is the typical order used in text mining:

`sentence split → word tokenize (regex) → lowercase → stop-word removal → POS tagging → lemmatization`

In [ ]:
def preprocess(text, stop_words=english_stops):
    tokenizer = RegexpTokenizer(r"[a-zA-Z']+")
    output = []
    for sent in sent_tokenize(text):                              # 1) sentence splitting
        tokens = tokenizer.tokenize(sent.lower())                 # 2) tokenization + lowercase
        tokens = [t for t in tokens if t not in stop_words]       # 3) stop-word removal
        for word, tag in nltk.pos_tag(tokens):                    # 4) POS tagging
            wn_tag = penn_to_wordnet(tag)
            output.append(lemmatizer.lemmatize(word, wn_tag) if wn_tag else word)   # 5) lemmatization
    return output

sample = ("Natural language processing is a field of computer science. "
          "Researchers are building models that understand languages better every year!")
print(preprocess(sample))

### ✅ What we practiced
1. Sentence splitting – `sent_tokenize` (language-specific Punkt models)
2. Word tokenization – `word_tokenize`, `WordPunctTokenizer`, `RegexpTokenizer`
3. Stop-word removal – NLTK list + custom list (always lowercase first)
4. Stemming (Porter, Lancaster) vs. lemmatization (WordNet, POS-aware)
5. POS tagging with the Penn Treebank tag set
6. NER with `ne_chunk`

Next notebook: **2. Visualization** – word-frequency graphs and word clouds on a real corpus.